# 01 · MLflow Basics — Flavor 기반 Logging + UC 등록 + Alias

## 이 노트북에서 배우는 것

1. **MLflow flavor**: sklearn 모델을 native flavor 로 logging
2. **Signature & Input example**: 자동 추론 + 의존성 캡처
3. **Unity Catalog 등록**: 3-level namespace (catalog.schema.model)
4. **Alias**: Champion / Challenger — 더이상 Staging/Production stage 없음
5. **모델 로드**: `models:/<name>@<alias>` URI 로 batch / serving 호출

In [ ]:
%run ./config

In [ ]:
import mlflow
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(experiment_path)

## Step 1. 데이터 로드 & Train/Test Split

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

pdf = spark.table(f"{catalog}.{schema}.customers").toPandas()

FEATURES = ["age", "tenure_months", "monthly_charges", "total_charges", "support_tickets"]
TARGET = "churned"

X_train, X_test, y_train, y_test = train_test_split(
    pdf[FEATURES], pdf[TARGET], test_size=0.2, stratify=pdf[TARGET], random_state=42
)
print(f"train={len(X_train)}, test={len(X_test)}")

## Step 2. 모델 학습 & MLflow Logging (Flavor 방식)

`mlflow.sklearn.log_model` 은 native flavor 로 직렬화하면서 `python_function` flavor도 자동 추가합니다.

**꼭 챙길 파라미터:**
- `signature` — 입력/출력 스키마. `infer_signature` 로 자동 생성
- `input_example` — 샘플 입력. **dependency auto-inference 트리거**
- `registered_model_name` — UC 3-level name, log + 등록을 한 번에

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from mlflow.models import infer_signature

with mlflow.start_run(run_name="rf_baseline") as run:
    # 1) 학습
    rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
    rf.fit(X_train, y_train)

    # 2) 평가
    preds = rf.predict(X_test)
    proba = rf.predict_proba(X_test)[:, 1]
    metrics = {
        "accuracy": accuracy_score(y_test, preds),
        "roc_auc":  roc_auc_score(y_test, proba),
    }
    mlflow.log_metrics(metrics)
    mlflow.log_params({"n_estimators": 100, "max_depth": 8})

    # 3) signature 자동 추론
    signature = infer_signature(X_train, rf.predict(X_train))

    # 4) log + UC 등록
    info = mlflow.sklearn.log_model(
        sk_model=rf,
        name="model",                       # MLflow 3.x: artifact_path → name
        signature=signature,
        input_example=X_train.iloc[:5],     # 의존성 추론 + 시그니처 검증 트리거
        registered_model_name=model_basic,  # log + register 한 번에
    )

print(f"metrics  : {metrics}")
print(f"model_uri: {info.model_uri}")
print(f"registered as: {model_basic}, version={info.registered_model_version}")

## Step 3. MLflow 가 생성한 파일들 살펴보기

`mlflow.sklearn.log_model` 호출 시 MLflow가 모델 디렉토리에 자동 생성하는 파일들:

```
model/
├── MLmodel              # 모델 메타데이터 (flavor, signature, env 참조)
├── model.pkl            # 직렬화된 sklearn 객체
├── python_env.yaml      # Python 버전 + build deps
├── requirements.txt     # pip 의존성 (auto-inferred)
├── conda.yaml           # conda env (fallback)
└── input_example.json   # 시그니처 검증용 샘플
```

In [ ]:
import os

local_path = mlflow.artifacts.download_artifacts(info.model_uri)
print("model artifact 구조:")
for f in sorted(os.listdir(local_path)):
    size = os.path.getsize(os.path.join(local_path, f))
    print(f"  {f:25s} {size:>10,} bytes")

### `requirements.txt` 자동 추론 결과 확인

In [ ]:
with open(os.path.join(local_path, "requirements.txt")) as f:
    print(f.read())

### `MLmodel` 메타데이터

In [ ]:
with open(os.path.join(local_path, "MLmodel")) as f:
    print(f.read())

## Step 4. Alias 설정 — UC 에서 Stage 대신 Alias 사용

Workspace Model Registry 의 `Staging` / `Production` stage 는 UC에서 **삭제**되었습니다.
대신 **mutable alias**(예: `Champion`, `Challenger`)와 **tag** 를 사용합니다.

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()
client.set_registered_model_alias(
    name=model_basic, alias="Champion", version=info.registered_model_version,
)
print(f"✓ {model_basic}@Champion → v{info.registered_model_version}")

## Step 5. Alias 로 모델 로드 & 예측

In [ ]:
# Alias 로 load
champion = mlflow.pyfunc.load_model(f"models:/{model_basic}@Champion")

# 단건 예측
sample = X_test.iloc[:3].reset_index(drop=True)
display(sample)

preds = champion.predict(sample)
print("predictions:", preds)

## Step 6. Spark UDF 로 배치 점수화

Model Serving 엔드포인트는 **low-latency real-time** 용입니다. 수백만 row 점수화는
`mlflow.pyfunc.spark_udf` 로 **클러스터에서 병렬 처리** 하세요.

In [ ]:
predict_udf = mlflow.pyfunc.spark_udf(
    spark,
    model_uri=f"models:/{model_basic}@Champion",
    env_manager="virtualenv",
)

scored = (
    spark.table(f"{catalog}.{schema}.customers")
         .limit(1000)
         .withColumn("score", predict_udf(*FEATURES))
)
display(scored.select("customer_id", *FEATURES, "score").limit(10))

## 정리

| 개념 | 핵심 |
| --- | --- |
| Flavor logging | `mlflow.sklearn.log_model(...)` — native 직렬화 + pyfunc 자동 등록 |
| signature/input_example | `infer_signature` + 5개 row 만 줘도 dep 추론 trigger |
| UC 등록 | `registered_model_name="catalog.schema.model"` 3-level |
| Alias | `client.set_registered_model_alias(..., "Champion", v)` |
| Load | `models:/<name>@<alias>` URI |
| Batch | `mlflow.pyfunc.spark_udf` 로 병렬 점수화 |

→ 다음: **`02_pyfunc_custom`** — PyFunc + load_context + artifacts 로 임의 코드 패키징